# Experiment B: Coverage Ablation

This notebook uses a talk-oriented synthetic proxy family with four latent factors:
amplitude, timing shift, position-like distortion, and shape-like distortion.
The position latent is synthetic because the current repo simulator does not expose detector position directly.

In [1]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from npml_support import *

dirs = ensure_npml_dirs()

In [2]:
out = run_coverage_ablation()
coverage_df = out["coverage_df"]
matrix_df = out["matrix_df"]

display(coverage_df)
display(matrix_df)

save_dataframe(coverage_df, "02_coverage_ablation_metrics.csv")
save_dataframe(matrix_df, "02_coverage_ablation_matrix.csv")

,training_condition,method,weighted_residual_mean,amplitude_rmse,timing_rmse,position_rmse,shape_rmse
0,full,EMPCA,2058.862959,0.176446,0.541410,0.340746,0.548300
1,full,PCA,2105.837498,0.198610,0.423035,0.544365,0.554438
2,timing_restricted,EMPCA,2060.238869,0.190101,0.540825,0.391622,0.552017
3,timing_restricted,PCA,2069.240298,0.197954,0.540825,0.524017,0.545023
4,position_restricted,EMPCA,2063.934786,0.204130,0.503432,0.561489,0.553370
5,position_restricted,PCA,2068.098922,0.201673,0.529250,0.561489,0.551420
6,shape_restricted,EMPCA,2059.150582,0.198795,0.510429,0.388023,0.550485
7,shape_restricted,PCA,2079.443261,0.209481,0.476585,0.536333,0.550485


,metric,correct_metric_full_coverage,wrong_metric_full_coverage,correct_metric_restricted_coverage,wrong_metric_restricted_coverage
0,weighted_residual,1.0,1.022816,1.002463,1.009996
1,amplitude_rmse,1.0,1.125610,1.156894,1.187224
2,timing_rmse,1.0,0.781359,0.998919,0.998919
3,position_rmse,1.0,1.597569,1.647824,1.647824


PosixPath('/Users/wongdowling/Documents/noise-weighted-subspace-reconstruction/paper2/npml/tables/02_coverage_ablation_matrix.csv')

In [3]:
metric_cols = ["amplitude_rmse", "timing_rmse", "position_rmse", "shape_rmse"]
ratio_df = coverage_df.copy()
full_empca = (
    coverage_df[(coverage_df["training_condition"] == "full") & (coverage_df["method"] == "EMPCA")]
    .iloc[0]
)
for col in metric_cols:
    ratio_df[col] = ratio_df[col] / max(full_empca[col], 1e-12)

empca_only = ratio_df[ratio_df["method"] == "EMPCA"].set_index("training_condition")[metric_cols]

fig, ax = plt.subplots(figsize=(7.5, 3.8))
im = ax.imshow(empca_only.to_numpy(), cmap="YlOrRd", aspect="auto")
ax.set_xticks(range(len(metric_cols)))
ax.set_xticklabels(metric_cols, rotation=20, ha="right")
ax.set_yticks(range(len(empca_only.index)))
ax.set_yticklabels(empca_only.index)
ax.set_title("Coverage failure when latent directions are omitted")
for i in range(empca_only.shape[0]):
    for j in range(empca_only.shape[1]):
        ax.text(j, i, f"{empca_only.iat[i, j]:.2f}x", ha="center", va="center", color="black")
fig.colorbar(im, ax=ax, shrink=0.8, label="relative error vs full-coverage EMPCA")
save_figure(fig, "02_coverage_ablation_heatmap.png")
plt.show()
plt.close(fig)

/var/folders/y6/2zvmjscx2s18w96j7qx32nnr0000gn/T/ipykernel_23419/3638411808.py:24: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [4]:
fig, ax = plt.subplots(figsize=(7.5, 4.2))
resid_plot = coverage_df.pivot(index="training_condition", columns="method", values="weighted_residual_mean")
resid_plot.plot(kind="bar", ax=ax, color=["#2b8a3e", "#c92a2a"])
ax.set_title("Weighted residual under full vs restricted coverage")
ax.set_ylabel("mean held-out chi2 proxy")
ax.legend(title="")
save_figure(fig, "02_coverage_ablation_weighted_residual.png")
plt.show()
plt.close(fig)

/var/folders/y6/2zvmjscx2s18w96j7qx32nnr0000gn/T/ipykernel_23419/690100569.py:8: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
